<a href="https://colab.research.google.com/github/mehmetbozdemir24/Magibu/blob/main/06_Tool_Call_with_sqlite_database/Tool_call_sqlite_database.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [41]:
from google.colab import drive
import os

drive.mount('/content/drive')

cwd = "/content/drive/MyDrive/Colab Notebooks/Magibu/"
os.makedirs(cwd, exist_ok=True)
DB_NAME = os.path.join(cwd, "bookstore.db")
print("Çalışma dizini hazır:", cwd)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Çalışma dizini hazır: /content/drive/MyDrive/Colab Notebooks/Magibu/


In [14]:
%%bash
# 1. Eksik zstd paketini yükle
apt-get update && apt-get install -y zstd

# 2. Ollama kurulumunu gerçekleştir
curl -fsSL https://ollama.com/install.sh | sh

# 3. Ollama sunucusunu arka planda başlat
nohup ollama serve > ollama.log 2>&1 &

# 4. Sunucunun hazır olması için bekle
sleep 5

# 5. Qwen3.5 9B modelini indir
ollama pull qwen3.5:9b

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [102 kB]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,861 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.6 MB]
Hit:13 https://ppa.launchpadcontent.net/graphics-driv

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling man

In [42]:
import os
import json
import sqlite3
from google.colab import drive
from tabulate import tabulate

# 1. DRIVE VE DİZİN YAPILANDIRMASI
drive.mount('/content/drive')

cwd = "/content/drive/MyDrive/Colab Notebooks/Magibu/"
os.makedirs(cwd, exist_ok=True)

# Bütün script boyunca TEK BİR veritabanı adı kullanıyoruz
DB_NAME = os.path.join(cwd, "kitapci.db")
print("Çalışma dizini ve veritabanı yolu hazır:", DB_NAME)

# 2. VERİTABANI İNİTİALİZASYONU
def init_db():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    # Kitaplar tablosu
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS books (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        title TEXT NOT NULL,
        author TEXT NOT NULL,
        genre TEXT NOT NULL,
        price REAL NOT NULL,
        stock INTEGER NOT NULL
    )
    """)

    # Siparişler tablosu
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS orders (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        book_id INTEGER,
        customer_name TEXT NOT NULL,
        quantity INTEGER NOT NULL,
        status TEXT DEFAULT 'Hazırlanıyor',
        FOREIGN KEY (book_id) REFERENCES books (id)
    )
    """)

    # Örnek kitap verileri
    cursor.execute("SELECT COUNT(*) FROM books")
    if cursor.fetchone()[0] == 0:
        sample_books = [
            ("Nutuk", "Mustafa Kemal Atatürk", "Tarih", 120.0, 15),
            ("Suç ve Ceza", "Dostoyevski", "Klasik", 95.0, 8),
            ("1984", "George Orwell", "Distopya", 85.0, 20),
            ("Yapay Zeka Çağı", "Kai-Fu Lee", "Teknoloji", 150.0, 5),
            ("Atomik Alışkanlıklar", "James Clear", "Kişisel Gelişim", 110.0, 12),
            ("Şeker Portakalı", "José Mauro de Vasconcelos", "Roman", 75.0, 18),
            ("Simyacı", "Paulo Coelho", "Roman", 80.0, 22),
            ("Kürk Mantolu Madonna", "Sabahattin Ali", "Klasik", 65.0, 30),
            ("Tutunamayanlar", "Oğuz Atay", "Roman", 140.0, 7),
            ("Saatleri Ayarlama Enstitüsü", "Ahmet Hamdi Tanpınar", "Klasik", 115.0, 10),
            ("Dönüşüm", "Franz Kafka", "Klasik", 50.0, 25),
            ("Fahrenheit 451", "Ray Bradbury", "Distopya", 90.0, 14),
            ("Sefiller", "Victor Hugo", "Klasik", 160.0, 9),
            ("Bülbülü Öldürmek", "Harper Lee", "Roman", 105.0, 11),
            ("Çalıkuşu", "Reşat Nuri Güntekin", "Klasik", 95.0, 16),
            ("İnce Memed", "Yaşar Kemal", "Roman", 130.0, 13),
            ("Doğunun Limanları", "Amin Maalouf", "Tarih", 88.0, 6),
            ("Beyaz Diş", "Jack London", "Macera", 60.0, 20),
            ("Küçük Prens", "Antoine de Saint-Exupéry", "Çocuk/Felsefe", 45.0, 40),
            ("Böyle Buyurdu Zerdüşt", "Friedrich Nietzsche", "Felsefe", 110.0, 8),
            ("Hayvan Çiftliği", "George Orwell", "Distopya", 70.0, 28),
            ("Yüzyıllık Yalnızlık", "Gabriel García Márquez", "Roman", 125.0, 10),
            ("Satranç", "Stefan Zweig", "Klasik", 40.0, 35),
            ("Körlük", "José Saramago", "Roman", 115.0, 9),
            ("Cesur Yeni Dünya", "Aldous Huxley", "Distopya", 95.0, 17),
            ("Sineklerin Tanrısı", "William Golding", "Roman", 85.0, 12),
            ("Kavgam", "Adolf Hitler", "Tarih", 130.0, 4),
            ("Genç Werther'in Acıları", "Goethe", "Klasik", 55.0, 19),
            ("Gargantua", "François Rabelais", "Mitoloji", 100.0, 5),
            ("Yeraltından Notlar", "Dostoyevski", "Klasik", 60.0, 24),
            ("Martin Eden", "Jack London", "Roman", 100.0, 15),
            ("Medyum", "Stephen King", "Korku", 135.0, 8),
            ("Otostopçunun Galaksi Rehberi", "Douglas Adams", "Bilim Kurgu", 90.0, 21),
            ("Vakıf", "Isaac Asimov", "Bilim Kurgu", 120.0, 11),
            ("Dune", "Frank Herbert", "Bilim Kurgu", 175.0, 14),
            ("Neuromancer", "William Gibson", "Bilim Kurgu", 110.0, 7),
            ("Cosmos", "Carl Sagan", "Bilim", 150.0, 10),
            ("Sapiens", "Yuval Noah Harari", "Tarih/Sosyoloji", 165.0, 25),
            ("Homo Deus", "Yuval Noah Harari", "Tarih/Felsefe", 160.0, 18),
            ("Tüfek, Mikrop ve Çelik", "Jared Diamond", "Tarih", 170.0, 6),
            ("İrade Terbiyesi", "Jules Payot", "Kişisel Gelişim", 70.0, 30),
            ("Düşün ve Zengin Ol", "Napoleon Hill", "Kişisel Gelişim", 85.0, 15),
            ("Derin Öğrenme", "Ian Goodfellow", "Teknoloji", 350.0, 3),
            ("Python ile Veri Analizi", "Wes McKinney", "Teknoloji", 220.0, 8),
            ("Temiz Kod (Clean Code)", "Robert C. Martin", "Teknoloji", 280.0, 10),
            ("Pragmatik Programcı", "Andrew Hunt", "Teknoloji", 260.0, 6),
            ("Aşk-ı Memnu", "Halid Ziya Uşaklıgil", "Klasik", 90.0, 12),
            ("Eylül", "Mehmet Rauf", "Klasik", 80.0, 14),
            ("Kar", "Orhan Pamuk", "Roman", 120.0, 9),
            ("Masumiyet Müzesi", "Orhan Pamuk", "Roman", 130.0, 11)
        ]
        cursor.executemany("INSERT INTO books (title, author, genre, price, stock) VALUES (?, ?, ?, ?, ?)", sample_books)

    # Örnek sipariş verileri (20 Müşteri)
    cursor.execute("SELECT COUNT(*) FROM orders")
    if cursor.fetchone()[0] == 0:
        sample_orders = [
            (1, "Ahmet Yılmaz", 2, "Kargoda"),
            (3, "Ayşe Kaya", 1, "Teslim Edildi"),
            (4, "Mehmet Demir", 1, "Hazırlanıyor"),
            (5, "Fatma Çelik", 3, "Kargoda"),
            (11, "Ali Öztürk", 1, "Teslim Edildi"),
            (15, "Zeynep Aydın", 2, "Hazırlanıyor"),
            (19, "Mustafa Arslan", 5, "Teslim Edildi"),
            (21, "Elif Doğan", 1, "Kargoda"),
            (23, "Burak Koç", 2, "Teslim Edildi"),
            (33, "Seda Kurt", 1, "Hazırlanıyor"),
            (35, "Emre Şahin", 1, "Kargoda"),
            (38, "Merve Özdemir", 2, "Teslim Edildi"),
            (43, "Can Yalçın", 1, "Hazırlanıyor"),
            (44, "Deniz Yıldız", 1, "Kargoda"),
            (45, "Oğuz Erdoğan", 2, "Teslim Edildi"),
            (2, "Selin Taş", 1, "Hazırlanıyor"),
            (8, "Kaan Kılıç", 1, "Teslim Edildi"),
            (10, "Büşra Bulut", 3, "Kargoda"),
            (16, "Cem Uncu", 1, "Hazırlanıyor"),
            (25, "Ece Tekin", 2, "Teslim Edildi")
        ]
        cursor.executemany("INSERT INTO orders (book_id, customer_name, quantity, status) VALUES (?, ?, ?, ?)", sample_orders)

    conn.commit()
    conn.close()

init_db()
print("Veritabanı tabloları ve varsayılan veriler başarıyla yüklendi.")

# 3. TABLO GÖRÜNTÜLEME FONKSİYONU
def show_database_tables():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    print("\n" + "=" * 60)
    print("📚 KİTAPLAR TABLOSU (books)")
    print("=" * 60)

    cursor.execute("SELECT id, title, author, genre, price, stock FROM books")
    books = cursor.fetchall()
    book_headers = ["ID", "Kitap Adı", "Yazar", "Tür", "Fiyat (₺)", "Stok"]
    print(tabulate(books, headers=book_headers, tablefmt="grid"))

    print("\n" + "=" * 60)
    print("🛒 SİPARİŞLER TABLOSU (orders)")
    print("=" * 60)

    cursor.execute("SELECT id, book_id, customer_name, quantity, status FROM orders")
    orders = cursor.fetchall()
    order_headers = ["Sipariş ID", "Kitap ID", "Müşteri Adı", "Adet", "Durum"]

    if orders:
        print(tabulate(orders, headers=order_headers, tablefmt="grid"))
    else:
        print("Henüz verilmiş bir sipariş bulunmuyor.")

    conn.close()

# Tabloları ekrana basıyoruz
show_database_tables()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Çalışma dizini ve veritabanı yolu hazır: /content/drive/MyDrive/Colab Notebooks/Magibu/kitapci.db
Veritabanı tabloları ve varsayılan veriler başarıyla yüklendi.

📚 KİTAPLAR TABLOSU (books)
+------+------------------------------+---------------------------+-----------------+-------------+--------+
|   ID | Kitap Adı                    | Yazar                     | Tür             |   Fiyat (₺) |   Stok |
+======+==============================+===========================+=================+=============+========+
|    1 | Nutuk                        | Mustafa Kemal Atatürk     | Tarih           |         120 |     13 |
+------+------------------------------+---------------------------+-----------------+-------------+--------+
|    2 | Suç ve Ceza                  | Dostoyevski               | Klasik          |          95 |      8 |
+------+--------------------

In [47]:
!pip install -q openai

import json
import sqlite3
from openai import OpenAI

# 1. MODEL ADI TANIMLAMASI
MODEL_NAME = "qwen3.5:9b" # Qwen/Qwen3.5-9B

# 2. TOOL FONKSİYONLARI
def get_books(genre: str = None) -> list:
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    if genre:
        cursor.execute("SELECT id, title, author, genre, price, stock FROM books WHERE genre LIKE ? AND stock > 0", (f"%{genre}%",))
    else:
        cursor.execute("SELECT id, title, author, genre, price, stock FROM books WHERE stock > 0")
    rows = cursor.fetchall()
    conn.close()
    return [{"id": r[0], "title": r[1], "author": r[2], "genre": r[3], "price": r[4], "stock": r[5]} for r in rows]

def create_order(book_id: int, quantity: int, customer_name: str) -> dict:
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute("SELECT stock, title FROM books WHERE id = ?", (book_id,))
    book = cursor.fetchone()

    if not book:
        conn.close()
        return {"success": False, "error": "Kitap bulunamadı."}

    stock, title = book
    if stock < quantity:
        conn.close()
        return {"success": False, "error": f"Yetersiz stok. Mevcut stok: {stock}"}

    cursor.execute("UPDATE books SET stock = stock - ? WHERE id = ?", (quantity, book_id))
    cursor.execute("INSERT INTO orders (book_id, customer_name, quantity) VALUES (?, ?, ?)", (book_id, customer_name, quantity))
    order_id = cursor.lastrowid

    conn.commit()
    conn.close()
    return {"success": True, "order_id": order_id, "message": f"'{title}' kitabından {quantity} adet sipariş alındı."}

def check_order_status(order_id: int) -> dict:
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute("SELECT o.id, b.title, o.customer_name, o.quantity, o.status FROM orders o JOIN books b ON o.book_id = b.id WHERE o.id = ?", (order_id,))
    row = cursor.fetchone()
    conn.close()

    if row:
        return {"order_id": row[0], "title": row[1], "customer": row[2], "quantity": row[3], "status": row[4]}
    return {"error": "Sipariş bulunamadı."}

# 3. SCHEMA VE MAP
TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "get_books",
            "description": "Stoktaki kitapları listeler. Kategoriye (genre) göre filtrelenebilir.",
            "parameters": {
                "type": "object",
                "properties": {"genre": {"type": "string", "description": "Kitap kategorisi (örn: Klasik, Bilim Kurgu)"}},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "create_order",
            "description": "Yeni bir kitap siparişi oluşturur.",
            "parameters": {
                "type": "object",
                "properties": {
                    "book_id": {"type": "integer", "description": "Kitabın ID'si"},
                    "quantity": {"type": "integer", "description": "Adet"},
                    "customer_name": {"type": "string", "description": "Müşteri adı"}
                },
                "required": ["book_id", "quantity", "customer_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "check_order_status",
            "description": "Sipariş durumunu sorgular.",
            "parameters": {
                "type": "object",
                "properties": {"order_id": {"type": "integer", "description": "Sipariş ID'si"}},
                "required": ["order_id"]
            }
        }
    }
]

TOOL_MAP = {
    "get_books": get_books,
    "create_order": create_order,
    "check_order_status": check_order_status
}

# 4. OPENAI CLIENT (OLLAMA LOCAL)
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

# 1. SYSTEM PROMPT GÜNCELLEMESİ
SYSTEM_PROMPT = """Sen dürüst ve yardımsever bir Online Kitapçı Asistanısın.
Kullanıcının isteklerini yanıtlamak için sana verilen fonksiyonları (tool call) kullanmalısın.

ÖNEMLİ KURALLAR:
1. Bir fonksiyon çalıştırıldıktan (tool response geldikten) sonra, çıkan sonucu analiz et ve kullanıcıya KESİNLİKLE Türkçe, nazik ve açıklayıcı bir yanıt ver.
2. Fonksiyon başarılı bir sipariş oluşturduysa, sipariş numarasını ve detaylarını kullanıcıya bildir.
3. Asla boş yanıt dönme."""


# 2. GÜNCELLENMİŞ RUN_AGENT FONKSİYONU
def run_agent(user_message: str, history: list):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    for h in history:
        messages.append({"role": "user", "content": h[0]})
        messages.append({"role": "assistant", "content": h[1]})

    messages.append({"role": "user", "content": user_message})

    # 1. Adım: Model Tool Çağıracak mı?
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        tools=TOOLS_SCHEMA,
        tool_choice="auto"
    )

    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls

    # Eğer model Tool çağırma kararı verdiyse:
    if tool_calls:
        messages.append(response_message)
        tool_executed_results = []

        for tool_call in tool_calls:
            func_name = tool_call.function.name
            func_args = json.loads(tool_call.function.arguments)
            call_id = tool_call.id if tool_call.id else "call_1"

            if func_name in TOOL_MAP:
                result = TOOL_MAP[func_name](**func_args)
                tool_executed_results.append(result)

                messages.append({
                    "role": "tool",
                    "tool_call_id": call_id,
                    "name": func_name,
                    "content": json.dumps(result, ensure_ascii=False)
                })

        # 2. Adım: Tool sonuçlarını modele verip nihai yanıtı isteme
        final_response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,
            temperature=0.3 # Bitiş yanıtını zorlamak için düşük sıcaklık
        )

        content = final_response.choices[0].message.content

        # Qwen3.5 boş yanıt dönerse Tool'dan dönen veriyi formatlayıp göster
        if not content or not content.strip():
            first_res = tool_executed_results[0] if tool_executed_results else {}
            if isinstance(first_res, dict):
                if first_res.get("success") and "message" in first_res:
                    return f"✅ Siparişiniz Alındı!\nSipariş No: **#{first_res.get('order_id')}**\n{first_res.get('message')}"
                elif "error" in first_res:
                    return f"❌ İşlem Başarısız: {first_res['error']}"
            return f"İşlem tamamlandı. Sonuç: {json.dumps(tool_executed_results, ensure_ascii=False)}"

        return content

    return response_message.content or "Bir yanıt oluşturulamadı."

In [48]:
print("--- TEST 1: Qwen/Qwen3.5-9B Kitap Sorgulama ---")
print(run_agent("Bilim Kurgu türündeki kitapları listeler misin?", []))

print("\n--- TEST 2: Sipariş Oluşturma ---")
print(run_agent("1 ID'li Nutuk kitabından Ali Kaya adına 2 adet sipariş vermek istiyorum.", []))

print("\n--- TEST 3: Sipariş Durumu Sorgulama ---")
print(run_agent("21 numaralı siparişimin durumu nedir?", []))

--- TEST 1: Qwen/Qwen3.5-9B Kitap Sorgulama ---
Elbette! İşte Bilim Kurgu türündeki kitaplarımız:

1. **Otostopçunun Galaksi Rehberi** - Douglas Adams  
   *Fiyat:* 90 TL | *Stok:* 21 adet  

2. **Vakıf** - Isaac Asimov  
   *Fiyat:* 120 TL | *Stok:* 11 adet  

3. **Dune** - Frank Herbert  
   *Fiyat:* 175 TL | *Stok:* 14 adet  

4. **Neuromancer** - William Gibson  
   *Fiyat:* 110 TL | *Stok:* 7 adet  

Bu kitaplardan birini beğendiyseniz, sipariş vermek için bana "sipariş ver" diyebilirsiniz! 😊

--- TEST 2: Sipariş Oluşturma ---
Harika! Siparişiniz başarıyla oluşturuldu. 📚

**Sipariş Detayları:**
- **Kitap Adı:** Nutuk
- **Miktar:** 2 adet
- **Ad Soyad:** Ali Kaya
- **Sipariş Numarası:** #22

Lütfen sipariş numaranızı (ID: 22) saklayın. Siparişiniz hazırlanmaya başlandı ve kısa süre içinde teslim edilecektir. Başka bir konuda yardıma ihtiyacınız olursa lütfen bana bildirin! 😊

--- TEST 3: Sipariş Durumu Sorgulama ---
Merhaba! Sipariş durumunuzu kontrol ettim ve aşağıdaki bilgilere u

In [ ]:
print("\n--- TEST 4: Veritabanında Olmayan Kitap (Halüsinasyon Testi) ---")
print(run_agent("Harry Potter kitabından 1 adet sipariş vermek istiyorum.", []))

print("\n--- TEST 5: Yetersiz Stok Testi ---")
# 43 ID'li 'Derin Öğrenme' kitabının stoğu 3 adet. 5 adet istenince stok hatası vermeli.
print(run_agent("43 ID'li Derin Öğrenme kitabından 5 adet almak istiyorum. Müşteri: Ahmet Demir", []))

print("\n--- TEST 6: Geçersiz Sipariş ID Sorgulama ---")
print(run_agent("9999 numaralı siparişimin durumu nedir?", []))

In [52]:
import subprocess
import time
import requests

def ensure_ollama_running():
    try:
        # Sunucunun açık olup olmadığını kontrol et
        res = requests.get("http://localhost:11434/")
        if res.status_code == 200:
            print("✅ Ollama sunucusu aktif ve çalışıyor.")
            return
    except Exception:
        print("⚠️ Ollama sunucusu kapalı tespit edildi, tekrar başlatılıyor...")

    # Sunucuyu arka planda başlat
    subprocess.Popen("nohup ollama serve > ollama.log 2>&1 &", shell=True)
    time.sleep(5)
    print("🚀 Ollama sunucusu başlatıldı.")

ensure_ollama_running()

⚠️ Ollama sunucusu kapalı tespit edildi, tekrar başlatılıyor...
🚀 Ollama sunucusu başlatıldı.


In [21]:
!pip install -q gradio

In [53]:
import gradio as gr

def predict(message, history):
    return run_agent(message, history)

demo = gr.ChatInterface(
    fn=predict,
    title="📚 Akıllı Kitapçı Asistanı (Qwen/Qwen3.5-9B Tool-Calling)",
    description="SQLite veritabanı ile entegre, stok kontrolü yapabilen ve sipariş alabilen yapay zeka asistanı.",
    examples=[
        "Teknoloji kategorisinde hangi kitaplar var?",
        "1 ID'li kitaptan Ahmet adına 1 adet sipariş ver.",
        "1 numaralı siparişimin durumu nedir?"
    ]
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a93e223c900f0f09e0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://a93e223c900f0f09e0.gradio.live
